# 49 — Region-Constrained Design Search v3

**Thesis:** topology specifies robustness; robustness specifies operation; operation specifies computation.

This notebook implements:

\[
D \rightarrow \Omega(D) \rightarrow \Omega_c(D) \rightarrow d(\partial\Omega_c) \rightarrow x^* \rightarrow \text{fault-tolerant execution}.
\]

The specification is the **region**. The operating point is one admitted realization inside it.

In [ ]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import ndimage

OUT = Path("outputs/notebook_49_v3")
FIG = OUT / "figures"
DATA = OUT / "data"
FIG.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 180, "font.size": 11})

def savefig(fig, name):
    path = FIG / name
    fig.savefig(path, bbox_inches="tight")
    return path

def clean_axes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)

def draw_box(ax, xy, wh, title, subtitle="", lw=2.0, title_size=11, subtitle_size=8):
    x,y = xy; w,h = wh
    ax.add_patch(Rectangle((x,y), w, h, fill=False, linewidth=lw, edgecolor="black"))
    ax.text(x+w/2, y+h*0.64, title, ha="center", va="center", fontsize=title_size, fontweight="bold")
    if subtitle:
        ax.text(x+w/2, y+h*0.28, subtitle, ha="center", va="center", fontsize=subtitle_size)

def arrow(ax, start, end, lw=2.0, color="black"):
    ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->", linewidth=lw, color=color, shrinkA=0, shrinkB=0))

## 1. Region-constrained design specification

Search over regions to preserve topology before selecting the operating point.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.2)); clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
labels=[("Design\n$D$","candidate\narchitecture"),("Admissible\nregion $\\Omega(D)$","thresholded\nspecification"),("Largest\ncomponent $\\Omega_c(D)$","admissible topology\n(specification)"),("Distance\ntransform","$d(\\partial\\Omega_c)$"),("Maximum-margin\npoint $x^*$","$\\arg\\max d$"),("Fault-tolerant\nexecution","robust logical\ncomputation")]
xs=np.linspace(0.05,0.86,len(labels)); w,h=0.12,0.34
for i,(t,s) in enumerate(labels):
    draw_box(ax,(xs[i],0.48),(w,h),t,s,lw=3.2 if "Largest" in t else 2.0,title_size=10,subtitle_size=8)
    if i < len(labels)-1: arrow(ax,(xs[i]+w+0.01,0.65),(xs[i+1]-0.012,0.65))
ax.text(0.28,0.22,"SPECIFICATION: determine what is admissible",ha="center",fontsize=9,fontweight="bold")
ax.text(0.70,0.22,"OPTIMIZATION: choose the best point within $\\Omega_c$",ha="center",fontsize=9,fontweight="bold")
ax.text(0.5,0.08,"Search over regions to preserve topology before selecting the operating point.",ha="center",fontsize=12)
ax.set_title("Region-Constrained Design Search Specification", fontsize=20)
savefig(fig,"49_v3_01_region_constrained_specification.png"); plt.show()

## 2. Objective-first vs specification-first search

A local objective can optimize the wrong thing. Specification-first search asks what remains connected before it chooses a point.

In [ ]:
fig, ax = plt.subplots(figsize=(12,5)); clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Objective-First vs Specification-First Search", fontsize=20)
ax.text(0.25,0.88,"Objective-first search",ha="center",fontsize=15,fontweight="bold")
for i,s in enumerate(["candidate","score / objective","gradient","iterate until convergence"]):
    y=0.70-i*0.15; draw_box(ax,(0.08,y),(0.34,0.08),s,title_size=10)
    if i<3: arrow(ax,(0.25,y),(0.25,y-0.06))
ax.text(0.25,0.12,"Can converge outside admissible topology\nand remain sensitive to local minima.",ha="center",fontsize=10)
ax.text(0.75,0.88,"Specification-first search",ha="center",fontsize=15,fontweight="bold")
for i,s in enumerate(["candidate design $D$","$\\Omega(D)$","$\\Omega_c(D)$","$d(\\partial\\Omega_c)$","$x^*$ maximum-margin point"]):
    y=0.70-i*0.12; draw_box(ax,(0.60,y),(0.30,0.07),s,title_size=10)
    if i<4: arrow(ax,(0.75,y),(0.75,y-0.05))
ax.text(0.75,0.12,"Operates on regions, preserves connected topology,\nand selects a robust operating point.",ha="center",fontsize=10)
ax.text(0.50,0.50,"vs.",ha="center",fontsize=18,fontweight="bold")
savefig(fig,"49_v3_02_objective_vs_specification_first.png"); plt.show()

## 3. Region objective

The objective rewards topology and robustness, then penalizes hardware cost and control complexity.

In [ ]:
fig, ax = plt.subplots(figsize=(13,5.2)); clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Region Objective: Engineering Specification", fontsize=20)
ax.add_patch(Rectangle((0.14,0.68),0.72,0.16,fill=False,linewidth=2.2))
ax.text(0.50,0.76,r"$J(D)=\alpha|\Omega_c(D)|+\beta M(D)-\gamma C(D)-\delta K(D)$",ha="center",va="center",fontsize=22)
for x,(t,s) in zip([0.12,0.32,0.56,0.76],[("+ |$\\Omega_c$|","maximize\nadmissible topology\n(region size)"),("+ M","maximize\nrobustness\nmargin"),("− C","minimize\nhardware\ncost"),("− K","minimize\ncontrol\ncomplexity")]):
    draw_box(ax,(x,0.30),(0.15,0.20),t,s,lw=2.0,title_size=16,subtitle_size=10)
ax.text(0.5,0.13,"First maximize $J(D)$ over designs. Then choose $x^*$ inside $\\Omega_c(D^*)$.",ha="center",fontsize=12)
savefig(fig,"49_v3_03_region_objective.png"); plt.show()

## 4. Synthetic admissible regions

A small reproducible model for drive × loss admissibility.

In [ ]:
N = 260
drive = np.linspace(0, 1, N)
loss = np.linspace(0, 1, N)
X, Y = np.meshgrid(drive, loss)

def sigmoid(z, s=0.055):
    return 1 / (1 + np.exp(-z / s))

def support_field(params):
    center = params.get("center", 0.62)
    width = params.get("width", 0.28)
    loss_scale = params.get("loss_scale", 0.20)
    plateau = sigmoid(X - (center - width), 0.045) * sigmoid((center + width) - X, 0.045)
    low_loss = np.exp(-(Y / loss_scale) ** 1.55)
    shoulder = 0.35 * np.exp(-((X - (center + 0.18))**2 / 0.025 + (Y - 0.10)**2 / 0.020))
    return np.clip(plateau * low_loss + shoulder, 0, 1)

designs = {
    "single cavity": {"center": 0.62, "width": 0.18, "loss_scale": 0.16, "cost": 2, "complexity": 2},
    "coupled resonators": {"center": 0.62, "width": 0.22, "loss_scale": 0.18, "cost": 4, "complexity": 4},
    "programmable mesh": {"center": 0.60, "width": 0.26, "loss_scale": 0.20, "cost": 7, "complexity": 7},
    "hybrid chip": {"center": 0.63, "width": 0.32, "loss_scale": 0.23, "cost": 9, "complexity": 8},
}
threshold = 0.36

def region_metrics(params):
    Z = support_field(params)
    mask = Z >= threshold
    labels, ncomp = ndimage.label(mask)
    if ncomp == 0:
        return dict(Z=Z, mask=mask, labels=labels, ncomp=0, largest=np.zeros_like(mask, dtype=bool),
                    dist=np.zeros_like(Z), admitted_area=0, largest_component=0, max_margin=0, average_margin=0,
                    x_star=np.nan, y_star=np.nan)
    sizes = ndimage.sum(mask, labels, index=np.arange(1, ncomp + 1))
    largest_id = int(np.argmax(sizes) + 1)
    largest = labels == largest_id
    dist = ndimage.distance_transform_edt(largest)
    dist_norm = dist / dist.max() if dist.max() else dist
    idx = np.unravel_index(np.argmax(dist), dist.shape)
    return dict(Z=Z, mask=mask, labels=labels, ncomp=int(ncomp), largest=largest, dist=dist_norm,
                admitted_area=float(mask.mean()), largest_component=float(largest.mean()),
                max_margin=float(dist.max()/N), average_margin=float(dist[largest].mean()/N),
                x_star=float(drive[idx[1]]), y_star=float(loss[idx[0]]))

rows, metrics = [], {}
for name, params in designs.items():
    m = region_metrics(params); metrics[name] = m
    score = 1.20*m["largest_component"] + 0.90*m["max_margin"] - 0.020*params["cost"] - 0.015*params["complexity"]
    rows.append({"design":name, "admitted_area":m["admitted_area"], "largest_component":m["largest_component"],
                 "component_count":m["ncomp"], "maximum_margin":m["max_margin"], "average_margin":m["average_margin"],
                 "hardware_cost":params["cost"], "control_complexity":params["complexity"], "score":score,
                 "x_star":m["x_star"], "y_star":m["y_star"]})
df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
df.to_csv(DATA / "49_v3_architecture_metrics.csv", index=False)
df.round(3)

## 5–7. Algorithm, design evolution, and distance transform

In [ ]:
fig, ax = plt.subplots(figsize=(8,6.2)); clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_title("Region-Constrained Search Algorithm", fontsize=20)
steps=[("Generate","candidate designs $D$"),("Evaluate","simulate performance field"),("Threshold","construct $\\Omega(D)$"),("Largest Component","extract $\\Omega_c(D)$"),("Distance Transform","compute $d(\\partial\\Omega_c)$"),("Region Objective","compute $J(D)$"),("Select","$D^*=\\arg\\max J(D)$; $x^*=\\arg\\max d$")]
for i,(t,s) in enumerate(steps):
    y=0.84-i*0.115; draw_box(ax,(0.20,y),(0.60,0.075),t,s,title_size=12,subtitle_size=8)
    ax.text(0.15,y+0.038,str(i+1),va="center",ha="center",fontsize=9,fontweight="bold")
    if i<len(steps)-1: arrow(ax,(0.50,y),(0.50,y-0.035))
savefig(fig,"49_v3_04_search_algorithm_overview.png"); plt.show()

stages=[("initial",{"center":0.62,"width":0.18,"loss_scale":0.16}),("admitted",{"center":0.62,"width":0.22,"loss_scale":0.18}),("connected",{"center":0.61,"width":0.27,"loss_scale":0.20}),("expanded\nlarger admissible topology",{"center":0.63,"width":0.34,"loss_scale":0.23})]
fig,axes=plt.subplots(1,4,figsize=(14,3.5),sharex=True,sharey=True)
for ax,(title,p) in zip(axes,stages):
    Z=support_field(p); mm=region_metrics(p)
    ax.imshow(Z,origin="lower",extent=[0,1,0,1],aspect="auto",vmin=0,vmax=1)
    ax.contour(X,Y,Z,levels=[threshold],colors="black",linewidths=1.6)
    ax.scatter([mm["x_star"]],[mm["y_star"]],s=45,zorder=4)
    ax.text(0.05,0.90,f"|Ωc|={mm['largest_component']:.2f}\nM={mm['max_margin']:.2f}",color="white",transform=ax.transAxes,fontsize=9,fontweight="bold")
    ax.set_title(title,fontsize=12); ax.set_xlabel("drive")
axes[0].set_ylabel("loss")
fig.suptitle("Design Evolution: Expand Admissible Topology Before Selecting $x^*$",fontsize=18)
savefig(fig,"49_v3_05_design_evolution.png"); plt.show()

best_name=df.loc[0,"design"]; best=metrics[best_name]; Z=best["Z"]; largest=best["largest"]; dist=best["dist"]
dist_masked=np.where(largest,dist,np.nan); x_star,y_star=best["x_star"],best["y_star"]
fig,ax=plt.subplots(figsize=(9,5.6))
im=ax.imshow(dist_masked,origin="lower",extent=[0,1,0,1],aspect="auto",vmin=0,vmax=1)
ax.contour(X,Y,Z,levels=[threshold],colors="black",linewidths=2.0)
ax.scatter([x_star],[y_star],s=160,zorder=5)
ax.annotate("$x^*$\nmaximum-margin point",xy=(x_star,y_star),xytext=(0.76,0.22),arrowprops=dict(arrowstyle="->",linewidth=2.0),fontsize=12)
ax.text(1.18,0.78,"distance transform\n$d(\\partial\\Omega_c)$\n\n↓\n\narg max\n\n↓\n\n$x^*$",transform=ax.transAxes,ha="center",va="center",fontsize=12)
ax.set_xlabel("drive"); ax.set_ylabel("loss"); ax.set_title("Distance Transform Selects $x^*$", fontsize=20)
fig.colorbar(im,ax=ax,label="normalized distance to failure boundary")
savefig(fig,"49_v3_06_distance_transform_selects_xstar.png"); plt.show()

## 8–9. Mathematical algorithm and specification table

In [ ]:
fig,ax=plt.subplots(figsize=(10,6.2)); clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_title("Algorithm: Search Over Regions, Not Points",fontsize=20)
pseudo="""for each candidate design D:

    Ω(D)  ← admissible_region(D)

    Ωc(D) ← largest_connected_component(Ω(D))

    M(D)  ← max_distance(Ωc(D), ∂Ωc(D))

    J(D)  ← α|Ωc(D)| + βM(D) − γC(D) − δK(D)

D* ← argmax_D J(D)

x* ← argmax_x∈Ωc(D*) d∂Ωc(D*)(x)
"""
ax.add_patch(Rectangle((0.08,0.10),0.84,0.75,fill=False,linewidth=2.2))
ax.text(0.13,0.78,pseudo,family="monospace",fontsize=14,va="top")
savefig(fig,"49_v3_07_mathematical_algorithm.png"); plt.show()

spec=pd.DataFrame([["Design D","candidate architecture or control policy","input to search"],["Ω(D)","admissible region","thresholded specification mask"],["Ωc(D)","largest connected admissible component","connected-component analysis"],["d(∂Ωc)","distance to failure boundary","distance transform"],["x*","maximum-margin operating point","argmax distance inside Ωc"],["J(D)","region objective","weighted engineering specification"],["Fault-tolerant computation","robust logical execution","selected from robust region"]],columns=["Symbol","Meaning","Computation"])
spec.to_csv(DATA/"49_v3_engineering_specification_table.csv",index=False)
fig,ax=plt.subplots(figsize=(13,4.2)); clean_axes(ax); ax.set_title("Engineering Specification for Region-Constrained Design Search",fontsize=20)
tbl=ax.table(cellText=spec.values,colLabels=spec.columns,cellLoc="center",loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.0,1.6)
for (r,c),cell in tbl.get_celld().items():
    cell.set_linewidth(1.0)
    if r==0: cell.set_text_props(fontweight="bold")
savefig(fig,"49_v3_08_engineering_specification_table.png"); plt.show()
spec

## 10–11. Search progress and architecture tradeoffs

In [ ]:
progress=pd.DataFrame({"stage":["initial","admitted","connected","expanded"],"|Ωc| region size":[0.11,0.13,0.16,0.20],"Margin M":[0.21,0.23,0.28,0.34]})
progress.to_csv(DATA/"49_v3_search_progress.csv",index=False)
fig,ax=plt.subplots(figsize=(10,5.2))
ax.plot(progress["stage"],progress["|Ωc| region size"],marker="o",linewidth=2.6,label="|Ωc| region size")
ax.plot(progress["stage"],progress["Margin M"],marker="s",linewidth=2.6,label="Margin M")
ax.fill_between(range(len(progress)),progress["|Ωc| region size"],progress["Margin M"],alpha=0.12)
ax.set_ylim(0,0.42); ax.set_ylabel("relative metric"); ax.set_title("Topology Improves Before Operating Margin",fontsize=20); ax.legend(); ax.grid(alpha=0.25)
savefig(fig,"49_v3_09_search_progress_topology_margin.png"); plt.show()

plot_df=df.copy()
plot_df["region"]=plot_df["largest_component"]/plot_df["largest_component"].max()
plot_df["robustness"]=plot_df["maximum_margin"]/plot_df["maximum_margin"].max()
plot_df["cost"]=plot_df["hardware_cost"]/plot_df["hardware_cost"].max()
plot_df["complexity"]=plot_df["control_complexity"]/plot_df["control_complexity"].max()
plot_df["score_norm"]=(plot_df["score"]-plot_df["score"].min())/(plot_df["score"].max()-plot_df["score"].min())
cols=["region","robustness","cost","complexity","score_norm"]; titles=["|Ωc| topology","robustness margin","hardware cost","control complexity","score"]
fig,axes=plt.subplots(1,len(cols),figsize=(15,4.3),sharey=True)
y=np.arange(len(plot_df))
for ax,col,title in zip(axes,cols,titles):
    ax.barh(y,plot_df[col]); ax.set_title(title,fontsize=11); ax.set_xlim(0,1.05); ax.grid(axis="x",alpha=0.25)
axes[0].set_yticks(y); axes[0].set_yticklabels(plot_df["design"]); axes[0].invert_yaxis()
fig.suptitle("Architecture Tradeoffs Sorted by Region-Constrained Score",fontsize=18)
savefig(fig,"49_v3_10_architecture_tradeoffs_sorted.png"); plt.show()

## 12–13. Pipeline and leading specification

In [ ]:
fig,ax=plt.subplots(figsize=(15,4.6)); clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_title("Region Search Pipeline",fontsize=20)
steps=[("Physics","what can exist?"),("Available\nResources","what was generated?"),("Admissibility","what survives?"),("Topology\n$\\Omega_c$","largest component"),("Distance\nGeometry","$d(\\partial\\Omega_c)$"),("Region\nObjective","$J(D)$"),("Operating\nPoint","$x^*$"),("Fault-Tolerant\nComputation","execution")]
xs=np.linspace(0.03,0.88,len(steps)); w,h=0.105,0.34
for i,(t,s) in enumerate(steps):
    draw_box(ax,(xs[i],0.40),(w,h),t,s,lw=2.1,title_size=10,subtitle_size=8)
    if i<len(steps)-1: arrow(ax,(xs[i]+w+0.006,0.57),(xs[i+1]-0.006,0.57),lw=1.8,color="0.2")
ax.text(0.5,0.15,"Topology specifies robustness. Robustness specifies operation. Operation specifies computation.",ha="center",fontsize=12)
savefig(fig,"49_v3_11_region_search_pipeline.png"); plt.show()

fig,ax=plt.subplots(figsize=(12,6.2)); clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_title("Topology Is the Leading Specification",fontsize=20)
ax.text(0.28,0.88,"Leading specification",ha="center",fontsize=15,fontweight="bold")
for i,item in enumerate(["Physics","Available resources","Admissibility","Topology $\\Omega_c$","Robustness","Operation $x^*$"]):
    y=0.76-i*0.105; draw_box(ax,(0.10,y),(0.36,0.07),item,title_size=10)
    if i<5: arrow(ax,(0.28,y),(0.28,y-0.035),lw=1.7)
ax.text(0.75,0.88,"Trailing optimization",ha="center",fontsize=15,fontweight="bold")
for i,item in enumerate(["Objective","Gradient","Local optimum","Summary metrics"]):
    y=0.68-i*0.13; draw_box(ax,(0.61,y),(0.28,0.07),item,lw=1.5,title_size=10)
    if i<3: arrow(ax,(0.75,y),(0.75,y-0.055),lw=1.6)
ax.text(0.5,0.08,"Region-constrained search treats topology as the specification and objectives as summaries.",ha="center",fontsize=12)
savefig(fig,"49_v3_12_topology_leading_specification.png"); plt.show()

## 14–16. Symbols, synthesis figure, and export summary

In [ ]:
symbol_summary=pd.DataFrame([["D","design","candidate architecture or policy"],["Ω(D)","admissible region","thresholded mask"],["Ωc(D)","largest connected component","connected-component analysis"],["d(∂Ωc)","distance to boundary","distance transform"],["x*","maximum-margin point","argmax inside Ωc"],["J(D)","region objective","α|Ωc| + βM − γC − δK"],["C(D)","hardware cost","estimated cost model"],["K(D)","control complexity","control / calibration model"],["M(D)","robustness margin","maximum distance in Ωc"]],columns=["Symbol","Meaning","Computation"])
symbol_summary.to_csv(DATA/"49_v3_symbol_summary.csv",index=False)
fig,ax=plt.subplots(figsize=(9,4.7)); clean_axes(ax); ax.set_title("Symbol Summary",fontsize=20)
tbl=ax.table(cellText=symbol_summary.values,colLabels=symbol_summary.columns,cellLoc="center",loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.0,1.45)
for (r,c),cell in tbl.get_celld().items():
    cell.set_linewidth(1.0)
    if r==0: cell.set_text_props(fontweight="bold")
savefig(fig,"49_v3_13_symbol_summary.png"); plt.show()
symbol_summary

fig,ax=plt.subplots(figsize=(14,4.4)); clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_title("Topology → Robustness → Operation",fontsize=20)
panel_x=[0.05,0.20,0.35,0.50,0.65,0.80]; titles=["candidate\nfield","threshold","$\\Omega$","$\\Omega_c$","$d(\\partial\\Omega_c)$","$x^*$"]
for i,x in enumerate(panel_x):
    ax.add_patch(Rectangle((x,0.35),0.10,0.25,fill=False,linewidth=1.5))
    ax.text(x+0.05,0.65,titles[i],ha="center",fontsize=9)
ax.imshow(support_field(designs["hybrid chip"]),origin="lower",extent=[panel_x[0],panel_x[0]+0.10,0.35,0.60],aspect="auto",vmin=0,vmax=1)
for px,lw in [(panel_x[1],1.6),(panel_x[2],1.6),(panel_x[3],2.6)]:
    ax.add_patch(Rectangle((px+0.025,0.38),0.05,0.16,fill=False,linewidth=lw))
ax.imshow(np.where(best["largest"],best["dist"],np.nan),origin="lower",extent=[panel_x[4],panel_x[4]+0.10,0.35,0.60],aspect="auto",vmin=0,vmax=1)
ax.imshow(np.where(best["largest"],best["dist"],np.nan),origin="lower",extent=[panel_x[5],panel_x[5]+0.10,0.35,0.60],aspect="auto",vmin=0,vmax=1)
ax.scatter([panel_x[5]+0.062],[0.38],s=30,color="red",zorder=5)
for i in range(len(panel_x)-1): arrow(ax,(panel_x[i]+0.11,0.475),(panel_x[i+1]-0.01,0.475),lw=1.8)
ax.text(0.25,0.18,"specify what is admissible",ha="center",fontsize=10,fontweight="bold")
ax.text(0.72,0.18,"optimize within the admissible topology",ha="center",fontsize=10,fontweight="bold")
savefig(fig,"49_v3_14_topology_robustness_operation.png"); plt.show()

summary={"notebook":"49_region_constrained_design_search_v3","thesis":"Topology specifies robustness; robustness specifies operation; operation specifies computation.","best_design":str(best_name),"figures":str(FIG),"data":str(DATA)}
(DATA/"49_v3_summary.json").write_text(json.dumps(summary,indent=2))
summary